# PICO Research Question Nanopublication Creator (Corrected)

Creates PICO nanopublications from a JSON configuration file.

**Template:** [Cochrane PICO Research Question](https://w3id.org/np/RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w)

This template uses the Cochrane PICO ontology (`http://data.cochrane.org/ontologies/pico/`).

---

## Instructions

1. **Create a JSON file** with your PICO details (see template below)
2. **Set the path** to your JSON file in Section 1
3. **Run All Cells** → Get your `.trig` file

---
# 📁 SECTION 1: INPUT FILE (EDIT THIS)
---

In [1]:
# Path to your PICO JSON file
PICO_FILE = "/Users/annef/Documents/ScienceLive/ai-agent/DGGS-AI-nanopubs/dggs_pico.json"
PICO_FILE = "/Users/annef/Documents/FAIR2Adapt/nanopub-notebooks/biodiversity/crete/crete_declaration_pico.json"

---
# ⚙️ SECTION 2: SETUP
---

In [2]:
# Install dependencies (uncomment if needed)
# !pip install nanopub rdflib

In [3]:
import json
import re
from rdflib import Graph, Dataset, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, XSD, FOAF
from datetime import datetime, timezone
from pathlib import Path

# Namespaces
NP = Namespace("http://www.nanopub.org/nschema#")
DCT = Namespace("http://purl.org/dc/terms/")
NT = Namespace("https://w3id.org/np/o/ntemplate/")
NPX = Namespace("http://purl.org/nanopub/x/")
PROV = Namespace("http://www.w3.org/ns/prov#")
ORCID = Namespace("https://orcid.org/")
PICO = Namespace("http://data.cochrane.org/ontologies/pico/")
SCIENCELIVE = Namespace("https://w3id.org/sciencelive/o/terms/")

# CORRECT PICO template URIs (Cochrane PICO ontology)
PICO_TEMPLATE = URIRef("https://w3id.org/np/RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w")

# Template references
PROV_TEMPLATE = URIRef("https://w3id.org/np/RA7lSq6MuK_TIC6JMSHvLtee3lpLoZDOqLJCLXevnrPoU")
PUBINFO_TEMPLATE_1 = URIRef("https://w3id.org/np/RA0J4vUn_dekg-U1kK3AOEt02p9mT2WO03uGxLDec1jLw")
PUBINFO_TEMPLATE_2 = URIRef("https://w3id.org/np/RAukAcWHRDlkqxk7H2XNSegc1WnHI569INvNr-xdptDGI")
PUBINFO_TEMPLATE_3 = URIRef("https://w3id.org/np/RAoTD7udB2KtUuOuAe74tJi1t3VzK0DyWS7rYVAq1GRvw")

# Question type mapping (Science Live URIs)
QUESTION_TYPE_MAP = {
    "causation": SCIENCELIVE.CausationResearchQuestion,
    "descriptive": SCIENCELIVE.DescriptiveResearchQuestion,
    "effectiveness": SCIENCELIVE.EffectivenessResearchQuestions,
    "experience": SCIENCELIVE.ExperienceResearchQuestions,
    "prediction": SCIENCELIVE.PredictionResearchQuestions,
}

VALID_QUESTION_TYPES = list(QUESTION_TYPE_MAP.keys())

print("✓ Setup complete")
print(f"  Template: {PICO_TEMPLATE}")

✓ Setup complete
  Template: https://w3id.org/np/RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w


---
# 📖 SECTION 3: LOAD & VALIDATE
---

In [4]:
# Load PICO from JSON
print(f"Loading: {PICO_FILE}")

with open(PICO_FILE, 'r', encoding='utf-8') as f:
    config = json.load(f)

# Extract fields
AUTHOR_ORCID = config['author']['orcid']
AUTHOR_NAME = config['author']['name']

TITLE = config['pico']['title']
POPULATION = config['pico']['population']
INTERVENTION = config['pico']['intervention']
COMPARISON = config['pico']['comparison']
OUTCOME = config['pico']['outcome']
RESEARCH_QUESTION = config['pico']['research_question']
QUESTION_TYPE = config['pico']['question_type']

OUTPUT_FILENAME = config['output']['filename']

print(f"✓ Loaded PICO: {TITLE[:50]}...")

Loading: /Users/annef/Documents/FAIR2Adapt/nanopub-notebooks/biodiversity/crete/crete_declaration_pico.json


JSONDecodeError: Expecting property name enclosed in double quotes: line 5 column 3 (char 84)

In [67]:
# Validate
print("Validating...")

errors = []
if not AUTHOR_ORCID:
    errors.append("author.orcid is required")
if not AUTHOR_NAME:
    errors.append("author.name is required")
if not TITLE or len(TITLE) < 10:
    errors.append("pico.title must be at least 10 characters")
if not POPULATION:
    errors.append("pico.population is required")
if not INTERVENTION:
    errors.append("pico.intervention is required")
if not OUTCOME:
    errors.append("pico.outcome is required")
if not RESEARCH_QUESTION:
    errors.append("pico.research_question is required")
if QUESTION_TYPE not in VALID_QUESTION_TYPES:
    errors.append(f"pico.question_type must be one of: {VALID_QUESTION_TYPES}")

if errors:
    print("❌ Validation errors:")
    for e in errors:
        print(f"   - {e}")
    raise ValueError("Please fix the errors in your JSON file")
else:
    print("✓ All fields valid")

Validating...
✓ All fields valid


---
# 🔨 SECTION 4: BUILD NANOPUBLICATION
---

In [68]:
# Create dataset with named graphs
TEMP_NP = Namespace("http://purl.org/nanopub/temp/np/")

this_np = URIRef("http://purl.org/nanopub/temp/np/")
head_graph = URIRef("http://purl.org/nanopub/temp/np/Head")
assertion_graph = URIRef("http://purl.org/nanopub/temp/np/assertion")
provenance_graph = URIRef("http://purl.org/nanopub/temp/np/provenance")
pubinfo_graph = URIRef("http://purl.org/nanopub/temp/np/pubinfo")

ds = Dataset()

# Bind prefixes
ds.bind("this", "http://purl.org/nanopub/temp/np/")
ds.bind("sub", "http://purl.org/nanopub/temp/np/")
ds.bind("np", NP)
ds.bind("dct", DCT)
ds.bind("nt", NT)
ds.bind("npx", NPX)
ds.bind("prov", PROV)
ds.bind("orcid", ORCID)
ds.bind("rdfs", RDFS)
ds.bind("xsd", XSD)
ds.bind("foaf", FOAF)
ds.bind("pico", PICO)
ds.bind("sciencelive", SCIENCELIVE)

print("✓ Dataset created")

✓ Dataset created


In [69]:
# Build Head graph
head = ds.graph(head_graph)
head.add((this_np, RDF.type, NP.Nanopublication))
head.add((this_np, NP.hasAssertion, assertion_graph))
head.add((this_np, NP.hasProvenance, provenance_graph))
head.add((this_np, NP.hasPublicationInfo, pubinfo_graph))

print(f"✓ Head: {len(head)} triples")

✓ Head: 4 triples


In [70]:
# Build Assertion graph using Cochrane PICO ontology
assertion = ds.graph(assertion_graph)

# Create a URI-safe slug from the title
def make_slug(text):
    """Convert title to URI-safe slug."""
    slug = text.lower()
    slug = re.sub(r'[^a-z0-9\s-]', '', slug)
    slug = re.sub(r'[\s_]+', '-', slug)
    slug = re.sub(r'-+', '-', slug)
    return slug[:80].strip('-')

pico_slug = make_slug(TITLE)
pico_uri = TEMP_NP[pico_slug]

# Create local resources for PICO components
population_uri = TEMP_NP["population"]
intervention_uri = TEMP_NP["interventionGroup"]
comparator_uri = TEMP_NP["comparatorGroup"]
outcome_uri = TEMP_NP["outcomeGroup"]

# Main PICO resource - type it as BOTH pico:PICO AND the Science Live question type
assertion.add((pico_uri, RDF.type, PICO.PICO))
question_type_uri = QUESTION_TYPE_MAP.get(QUESTION_TYPE)
if question_type_uri:
    assertion.add((pico_uri, RDF.type, question_type_uri))

# Label and description
assertion.add((pico_uri, RDFS.label, Literal(TITLE)))
assertion.add((pico_uri, DCT.description, Literal(RESEARCH_QUESTION)))

# PICO components using Cochrane PICO predicates
# Population
assertion.add((pico_uri, PICO.population, population_uri))
assertion.add((population_uri, DCT.description, Literal(POPULATION)))

# Intervention
assertion.add((pico_uri, PICO.interventionGroup, intervention_uri))
assertion.add((intervention_uri, DCT.description, Literal(INTERVENTION)))

# Comparator
assertion.add((pico_uri, PICO.comparatorGroup, comparator_uri))
assertion.add((comparator_uri, DCT.description, Literal(COMPARISON)))

# Outcome
assertion.add((pico_uri, PICO.outcomeGroup, outcome_uri))
assertion.add((outcome_uri, DCT.description, Literal(OUTCOME)))

print(f"✓ Assertion: {len(assertion)} triples")
print(f"  PICO URI: {pico_uri}")

✓ Assertion: 12 triples
  PICO URI: http://purl.org/nanopub/temp/np/dggs-based-land-use-classification-performance-evaluation


In [71]:
# Build Provenance graph
provenance = ds.graph(provenance_graph)
author_uri = ORCID[AUTHOR_ORCID]
provenance.add((assertion_graph, PROV.wasAttributedTo, author_uri))

print(f"✓ Provenance: {len(provenance)} triples")

✓ Provenance: 1 triples


In [72]:
# Build Publication Info graph
pubinfo = ds.graph(pubinfo_graph)

# Creator info
pubinfo.add((author_uri, FOAF.name, Literal(AUTHOR_NAME)))

# Nanopub metadata
timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00")
pubinfo.add((this_np, DCT.created, Literal(timestamp, datatype=XSD.dateTime)))
pubinfo.add((this_np, DCT.creator, author_uri))
pubinfo.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
pubinfo.add((this_np, NPX.wasCreatedAt, URIRef("https://platform.sciencelive4all.org/")))

# CRITICAL: npx:introduces enables federated SPARQL queries to find this resource
pubinfo.add((this_np, NPX.introduces, pico_uri))

# Label (truncate if needed)
label = f"PICO Research Question: {TITLE}"
if len(label) > 100:
    label = label[:97] + "..."
pubinfo.add((this_np, RDFS.label, Literal(label)))

# Template references (CORRECT template)
pubinfo.add((this_np, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_1))
pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_2))
pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_3))
pubinfo.add((this_np, NT.wasCreatedFromTemplate, PICO_TEMPLATE))

print(f"✓ Pubinfo: {len(pubinfo)} triples")

✓ Pubinfo: 12 triples


---
# 📄 SECTION 5: OUTPUT
---

In [73]:
# Serialize and save
trig_output = ds.serialize(format="trig")

output_path = Path(f"{OUTPUT_FILENAME}.trig")
with open(output_path, "w", encoding="utf-8") as f:
    f.write(trig_output)

print(f"✓ Saved to: {output_path.absolute()}")

✓ Saved to: /Users/annef/Documents/FAIR2Adapt/nanopub-notebooks/notebooks/dggs-landuse-pico.trig


In [74]:
# Display output
print("=" * 70)
print("NANOPUBLICATION (TriG format)")
print("=" * 70)
print(trig_output)

NANOPUBLICATION (TriG format)
@prefix dct: <http://purl.org/dc/terms/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix np: <http://www.nanopub.org/nschema#> .
@prefix npx: <http://purl.org/nanopub/x/> .
@prefix nt: <https://w3id.org/np/o/ntemplate/> .
@prefix orcid: <https://orcid.org/> .
@prefix pico: <http://data.cochrane.org/ontologies/pico/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sciencelive: <https://w3id.org/sciencelive/o/terms/> .
@prefix sub: <http://purl.org/nanopub/temp/np/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

sub:pubinfo {
    sub: rdfs:label "PICO Research Question: DGGS-based Land-Use Classification Performance Evaluation" ;
        dct:created "2026-01-30T13:13:45+00:00"^^xsd:dateTime ;
        dct:creator orcid:0000-0002-7400-2530 ;
        dct:license <https://creativecommons.org/licenses/by/4.0/> ;
        npx:introduces sub:dggs-based-land-use-classification-performance-ev

In [75]:
# Summary
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Input:    {PICO_FILE}")
print(f"Output:   {output_path}")
print(f"Author:   {AUTHOR_NAME} (orcid:{AUTHOR_ORCID})")
print(f"Type:     {QUESTION_TYPE}")
print(f"Template: RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w (Cochrane PICO)")
print()
print("PICO:")
print(f"  Title: {TITLE[:60]}..." if len(TITLE) > 60 else f"  Title: {TITLE}")
print(f"  P: {POPULATION[:55]}..." if len(POPULATION) > 55 else f"  P: {POPULATION}")
print(f"  I: {INTERVENTION[:55]}..." if len(INTERVENTION) > 55 else f"  I: {INTERVENTION}")
print(f"  C: {COMPARISON[:55]}..." if len(COMPARISON) > 55 else f"  C: {COMPARISON}")
print(f"  O: {OUTCOME[:55]}..." if len(OUTCOME) > 55 else f"  O: {OUTCOME}")
print()
print("Next steps:")
print(f"  Sign:    nanopub sign {output_path}")
print(f"  Publish: nanopub publish {output_path.stem}.signed.trig")

SUMMARY
Input:    /Users/annef/Documents/ScienceLive/ai-agent/DGGS-AI-nanopubs/dggs_pico.json
Output:   dggs-landuse-pico.trig
Author:   Richard M. Law (orcid:0000-0002-7400-2530)
Type:     effectiveness
Template: RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w (Cochrane PICO)

PICO:
  Title: DGGS-based Land-Use Classification Performance Evaluation
  P: Geospatial data processing workflows for land-use class...
  I: Discrete Global Grid Systems (DGGS) using the H3 hexago...
  C: Traditional vector-based workflows (spatial union opera...
  O: Computational performance (processing time, scalability...

Next steps:
  Sign:    nanopub sign dggs-landuse-pico.trig
  Publish: nanopub publish dggs-landuse-pico.signed.trig


---
# 🚀 SECTION 6: SIGN & PUBLISH (OPTIONAL)
---

In [76]:
PUBLISH = True
USE_TEST_SERVER = False
PROFILE_PATH = "/Users/annef/Documents/ScienceLive/ai-profile/profile.yml"  # Set to your profile.yml path

In [77]:
if PUBLISH:
    from nanopub import Nanopub, NanopubConf, load_profile
    
    if PROFILE_PATH:
        profile = load_profile(PROFILE_PATH)
        print(f"Loaded profile: {profile.name}")
    else:
        profile = load_profile()  # Uses default profile
        print(f"Using default profile")
    
    conf = NanopubConf(profile=profile, use_test_server=USE_TEST_SERVER)
    np_obj = Nanopub(rdf=output_path, conf=conf)
    
    np_obj.sign()
    print(f"✓ Signed")
    
    signed_path = Path(f"{OUTPUT_FILENAME}.signed.trig")
    np_obj.store(signed_path)
    print(f"✓ Saved: {signed_path}")
    print(f"Actually saved to: {signed_path.absolute()}")
    
    # Uncomment to publish:
#    np_obj.publish()
    print(f"✓ Published: {np_obj.source_uri}")
else:
    print("Publishing disabled. Set PUBLISH = True to enable.")

Loaded profile: claude-ai-agent
✓ Signed
✓ Saved: dggs-landuse-pico.signed.trig
Actually saved to: /Users/annef/Documents/FAIR2Adapt/nanopub-notebooks/notebooks/dggs-landuse-pico.signed.trig
✓ Published: https://w3id.org/np/RABIJ_iwhASJUw9uKDZnaSNL_IH3JW1REwAmKUzfca2MY


---
# 📋 JSON TEMPLATE

Create a JSON file with this structure:

```json
{
  "author": {
    "orcid": "0000-0000-0000-0000",
    "name": "Your Name"
  },
  "pico": {
    "title": "Your systematic review title",
    "population": "Who or what is being studied",
    "intervention": "What intervention or exposure",
    "comparison": "Comparison group (or 'Not applicable')",
    "outcome": "What outcomes are measured",
    "research_question": "Your full research question",
    "question_type": "effectiveness"
  },
  "output": {
    "filename": "my-pico"
  }
}
```

**Question types:** `causation`, `descriptive`, `effectiveness`, `experience`, `prediction`

**Note:** The `evaluation` type is NOT available in the Cochrane PICO template.

---

# PICO Research Question Nanopublication Creator (Corrected)

Creates PICO nanopublications from a JSON configuration file.

**Template:** [Cochrane PICO Research Question](https://w3id.org/np/RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w)

This template uses the Cochrane PICO ontology (`http://data.cochrane.org/ontologies/pico/`).

---

## Instructions

1. **Create a JSON file** with your PICO details (see template below)
2. **Set the path** to your JSON file in Section 1
3. **Run All Cells** → Get your `.trig` file

---
# 📁 SECTION 1: INPUT FILE (EDIT THIS)
---

In [1]:
# Path to your PICO JSON file
PICO_FILE = "/Users/annef/Documents/ScienceLive/ai-agent/DGGS-AI-nanopubs/dggs_pico.json"

---
# ⚙️ SECTION 2: SETUP
---

In [2]:
# Install dependencies (uncomment if needed)
# !pip install nanopub rdflib

In [3]:
import json
import re
from rdflib import Graph, Dataset, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, XSD, FOAF
from datetime import datetime, timezone
from pathlib import Path

# Namespaces
NP = Namespace("http://www.nanopub.org/nschema#")
DCT = Namespace("http://purl.org/dc/terms/")
NT = Namespace("https://w3id.org/np/o/ntemplate/")
NPX = Namespace("http://purl.org/nanopub/x/")
PROV = Namespace("http://www.w3.org/ns/prov#")
ORCID = Namespace("https://orcid.org/")
PICO = Namespace("http://data.cochrane.org/ontologies/pico/")
SCIENCELIVE = Namespace("https://w3id.org/sciencelive/o/terms/")

# CORRECT PICO template URIs (Cochrane PICO ontology)
PICO_TEMPLATE = URIRef("https://w3id.org/np/RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w")

# Template references
PROV_TEMPLATE = URIRef("https://w3id.org/np/RA7lSq6MuK_TIC6JMSHvLtee3lpLoZDOqLJCLXevnrPoU")
PUBINFO_TEMPLATE_1 = URIRef("https://w3id.org/np/RA0J4vUn_dekg-U1kK3AOEt02p9mT2WO03uGxLDec1jLw")
PUBINFO_TEMPLATE_2 = URIRef("https://w3id.org/np/RAukAcWHRDlkqxk7H2XNSegc1WnHI569INvNr-xdptDGI")
PUBINFO_TEMPLATE_3 = URIRef("https://w3id.org/np/RAoTD7udB2KtUuOuAe74tJi1t3VzK0DyWS7rYVAq1GRvw")

# Question type mapping (Science Live URIs)
QUESTION_TYPE_MAP = {
    "causation": SCIENCELIVE.CausationResearchQuestion,
    "descriptive": SCIENCELIVE.DescriptiveResearchQuestion,
    "effectiveness": SCIENCELIVE.EffectivenessResearchQuestions,
    "experience": SCIENCELIVE.ExperienceResearchQuestions,
    "prediction": SCIENCELIVE.PredictionResearchQuestions,
}

VALID_QUESTION_TYPES = list(QUESTION_TYPE_MAP.keys())

print("✓ Setup complete")
print(f"  Template: {PICO_TEMPLATE}")

✓ Setup complete
  Template: https://w3id.org/np/RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w


---
# 📖 SECTION 3: LOAD & VALIDATE
---

In [4]:
# Load PICO from JSON
print(f"Loading: {PICO_FILE}")

with open(PICO_FILE, 'r', encoding='utf-8') as f:
    config = json.load(f)

# Extract fields
AUTHOR_ORCID = config['author']['orcid']
AUTHOR_NAME = config['author']['name']

TITLE = config['pico']['title']
POPULATION = config['pico']['population']
INTERVENTION = config['pico']['intervention']
COMPARISON = config['pico']['comparison']
OUTCOME = config['pico']['outcome']
RESEARCH_QUESTION = config['pico']['research_question']
QUESTION_TYPE = config['pico']['question_type']

OUTPUT_FILENAME = config['output']['filename']

print(f"✓ Loaded PICO: {TITLE[:50]}...")

Loading: /Users/annef/Documents/ScienceLive/ai-agent/DGGS-AI-nanopubs/dggs_pico.json
✓ Loaded PICO: DGGS-based Land-Use Classification Performance Eva...


In [5]:
# Validate
print("Validating...")

errors = []
if not AUTHOR_ORCID:
    errors.append("author.orcid is required")
if not AUTHOR_NAME:
    errors.append("author.name is required")
if not TITLE or len(TITLE) < 10:
    errors.append("pico.title must be at least 10 characters")
if not POPULATION:
    errors.append("pico.population is required")
if not INTERVENTION:
    errors.append("pico.intervention is required")
if not OUTCOME:
    errors.append("pico.outcome is required")
if not RESEARCH_QUESTION:
    errors.append("pico.research_question is required")
if QUESTION_TYPE not in VALID_QUESTION_TYPES:
    errors.append(f"pico.question_type must be one of: {VALID_QUESTION_TYPES}")

if errors:
    print("❌ Validation errors:")
    for e in errors:
        print(f"   - {e}")
    raise ValueError("Please fix the errors in your JSON file")
else:
    print("✓ All fields valid")

Validating...
✓ All fields valid


---
# 🔨 SECTION 4: BUILD NANOPUBLICATION
---

In [6]:
# Create dataset with named graphs
TEMP_NP = Namespace("http://purl.org/nanopub/temp/np/")

this_np = URIRef("http://purl.org/nanopub/temp/np/")
head_graph = URIRef("http://purl.org/nanopub/temp/np/Head")
assertion_graph = URIRef("http://purl.org/nanopub/temp/np/assertion")
provenance_graph = URIRef("http://purl.org/nanopub/temp/np/provenance")
pubinfo_graph = URIRef("http://purl.org/nanopub/temp/np/pubinfo")

ds = Dataset()

# Bind prefixes
ds.bind("this", "http://purl.org/nanopub/temp/np/")
ds.bind("sub", "http://purl.org/nanopub/temp/np/")
ds.bind("np", NP)
ds.bind("dct", DCT)
ds.bind("nt", NT)
ds.bind("npx", NPX)
ds.bind("prov", PROV)
ds.bind("orcid", ORCID)
ds.bind("rdfs", RDFS)
ds.bind("xsd", XSD)
ds.bind("foaf", FOAF)
ds.bind("pico", PICO)
ds.bind("sciencelive", SCIENCELIVE)

print("✓ Dataset created")

✓ Dataset created


In [7]:
# Build Head graph
head = ds.graph(head_graph)
head.add((this_np, RDF.type, NP.Nanopublication))
head.add((this_np, NP.hasAssertion, assertion_graph))
head.add((this_np, NP.hasProvenance, provenance_graph))
head.add((this_np, NP.hasPublicationInfo, pubinfo_graph))

print(f"✓ Head: {len(head)} triples")

✓ Head: 4 triples


In [8]:
# Build Assertion graph using Cochrane PICO ontology
assertion = ds.graph(assertion_graph)

# Create a URI-safe slug from the title
def make_slug(text):
    """Convert title to URI-safe slug."""
    slug = text.lower()
    slug = re.sub(r'[^a-z0-9\s-]', '', slug)
    slug = re.sub(r'[\s_]+', '-', slug)
    slug = re.sub(r'-+', '-', slug)
    return slug[:80].strip('-')

pico_slug = make_slug(TITLE)
pico_uri = TEMP_NP[pico_slug]

# Create local resources for PICO components
population_uri = TEMP_NP["population"]
intervention_uri = TEMP_NP["interventionGroup"]
comparator_uri = TEMP_NP["comparatorGroup"]
outcome_uri = TEMP_NP["outcomeGroup"]

# Main PICO resource - type it as BOTH pico:PICO AND the Science Live question type
assertion.add((pico_uri, RDF.type, PICO.PICO))
question_type_uri = QUESTION_TYPE_MAP.get(QUESTION_TYPE)
if question_type_uri:
    assertion.add((pico_uri, RDF.type, question_type_uri))

# Label and description
assertion.add((pico_uri, RDFS.label, Literal(TITLE)))
assertion.add((pico_uri, DCT.description, Literal(RESEARCH_QUESTION)))

# PICO components using Cochrane PICO predicates
# Population
assertion.add((pico_uri, PICO.population, population_uri))
assertion.add((population_uri, DCT.description, Literal(POPULATION)))

# Intervention
assertion.add((pico_uri, PICO.interventionGroup, intervention_uri))
assertion.add((intervention_uri, DCT.description, Literal(INTERVENTION)))

# Comparator
assertion.add((pico_uri, PICO.comparatorGroup, comparator_uri))
assertion.add((comparator_uri, DCT.description, Literal(COMPARISON)))

# Outcome
assertion.add((pico_uri, PICO.outcomeGroup, outcome_uri))
assertion.add((outcome_uri, DCT.description, Literal(OUTCOME)))

print(f"✓ Assertion: {len(assertion)} triples")
print(f"  PICO URI: {pico_uri}")

✓ Assertion: 12 triples
  PICO URI: http://purl.org/nanopub/temp/np/dggs-based-land-use-classification-performance-evaluation


In [9]:
# Build Provenance graph
provenance = ds.graph(provenance_graph)
author_uri = ORCID[AUTHOR_ORCID]
provenance.add((assertion_graph, PROV.wasAttributedTo, author_uri))

print(f"✓ Provenance: {len(provenance)} triples")

✓ Provenance: 1 triples


In [10]:
# Build Publication Info graph
pubinfo = ds.graph(pubinfo_graph)

# Creator info
pubinfo.add((author_uri, FOAF.name, Literal(AUTHOR_NAME)))

# Nanopub metadata
timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00")
pubinfo.add((this_np, DCT.created, Literal(timestamp, datatype=XSD.dateTime)))
pubinfo.add((this_np, DCT.creator, author_uri))
pubinfo.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
pubinfo.add((this_np, NPX.wasCreatedAt, URIRef("https://nanodash.knowledgepixels.com/")))

# CRITICAL: npx:introduces enables federated SPARQL queries to find this resource
pubinfo.add((this_np, NPX.introduces, pico_uri))

# Label (truncate if needed)
label = f"PICO Research Question: {TITLE}"
if len(label) > 100:
    label = label[:97] + "..."
pubinfo.add((this_np, RDFS.label, Literal(label)))

# Template references (CORRECT template)
pubinfo.add((this_np, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_1))
pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_2))
pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_3))
pubinfo.add((this_np, NT.wasCreatedFromTemplate, PICO_TEMPLATE))

print(f"✓ Pubinfo: {len(pubinfo)} triples")

✓ Pubinfo: 12 triples


---
# 📄 SECTION 5: OUTPUT
---

In [11]:
# Serialize and save
trig_output = ds.serialize(format="trig")

output_path = Path(f"{OUTPUT_FILENAME}.trig")
with open(output_path, "w", encoding="utf-8") as f:
    f.write(trig_output)

print(f"✓ Saved to: {output_path.absolute()}")

✓ Saved to: /Users/annef/Documents/FAIR2Adapt/nanopub-notebooks/notebooks/dggs-landuse-pico.trig


In [12]:
# Display output
print("=" * 70)
print("NANOPUBLICATION (TriG format)")
print("=" * 70)
print(trig_output)

NANOPUBLICATION (TriG format)
@prefix dct: <http://purl.org/dc/terms/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix np: <http://www.nanopub.org/nschema#> .
@prefix npx: <http://purl.org/nanopub/x/> .
@prefix nt: <https://w3id.org/np/o/ntemplate/> .
@prefix orcid: <https://orcid.org/> .
@prefix pico: <http://data.cochrane.org/ontologies/pico/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sciencelive: <https://w3id.org/sciencelive/o/terms/> .
@prefix sub: <http://purl.org/nanopub/temp/np/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

sub:pubinfo {
    sub: rdfs:label "PICO Research Question: DGGS-based Land-Use Classification Performance Evaluation" ;
        dct:created "2026-01-29T13:49:53+00:00"^^xsd:dateTime ;
        dct:creator orcid:0000-0002-1784-2920 ;
        dct:license <https://creativecommons.org/licenses/by/4.0/> ;
        npx:introduces sub:dggs-based-land-use-classification-performance-ev

In [13]:
# Summary
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Input:    {PICO_FILE}")
print(f"Output:   {output_path}")
print(f"Author:   {AUTHOR_NAME} (orcid:{AUTHOR_ORCID})")
print(f"Type:     {QUESTION_TYPE}")
print(f"Template: RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w (Cochrane PICO)")
print()
print("PICO:")
print(f"  Title: {TITLE[:60]}..." if len(TITLE) > 60 else f"  Title: {TITLE}")
print(f"  P: {POPULATION[:55]}..." if len(POPULATION) > 55 else f"  P: {POPULATION}")
print(f"  I: {INTERVENTION[:55]}..." if len(INTERVENTION) > 55 else f"  I: {INTERVENTION}")
print(f"  C: {COMPARISON[:55]}..." if len(COMPARISON) > 55 else f"  C: {COMPARISON}")
print(f"  O: {OUTCOME[:55]}..." if len(OUTCOME) > 55 else f"  O: {OUTCOME}")
print()
print("Next steps:")
print(f"  Sign:    nanopub sign {output_path}")
print(f"  Publish: nanopub publish {output_path.stem}.signed.trig")

SUMMARY
Input:    /Users/annef/Documents/ScienceLive/ai-agent/DGGS-AI-nanopubs/dggs_pico.json
Output:   dggs-landuse-pico.trig
Author:   Anne Fouilloux (orcid:0000-0002-1784-2920)
Type:     effectiveness
Template: RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w (Cochrane PICO)

PICO:
  Title: DGGS-based Land-Use Classification Performance Evaluation
  P: Geospatial data processing workflows for land-use class...
  I: Discrete Global Grid Systems (DGGS) using the H3 hexago...
  C: Traditional vector-based workflows (spatial union opera...
  O: Computational performance (processing time, scalability...

Next steps:
  Sign:    nanopub sign dggs-landuse-pico.trig
  Publish: nanopub publish dggs-landuse-pico.signed.trig


---
# 🚀 SECTION 6: SIGN & PUBLISH (OPTIONAL)
---

In [14]:
PUBLISH = True
USE_TEST_SERVER = False
PROFILE_PATH = "/Users/annef/Documents/ScienceLive/ai-profile/profile.yml"  # Set to your profile.yml path

In [15]:
if PUBLISH:
    from nanopub import Nanopub, NanopubConf, load_profile
    
    if PROFILE_PATH:
        profile = load_profile(PROFILE_PATH)
        print(f"Loaded profile: {profile.name}")
    else:
        profile = load_profile()  # Uses default profile
        print(f"Using default profile")
    
    conf = NanopubConf(profile=profile, use_test_server=USE_TEST_SERVER)
    np_obj = Nanopub(rdf=output_path, conf=conf)
    
    np_obj.sign()
    print(f"✓ Signed")
    
    signed_path = Path(f"{OUTPUT_FILENAME}.signed.trig")
    np_obj.store(signed_path)
    print(f"✓ Saved: {signed_path}")
    print(f"Actually saved to: {signed_path.absolute()}")
    
    # Uncomment to publish:
    # np_obj.publish()
    # print(f"✓ Published: {np_obj.source_uri}")
else:
    print("Publishing disabled. Set PUBLISH = True to enable.")

ProfileError: An error occurred:
  in "/Users/annef/Documents/ScienceLive/ai-profile/profile.yml", line 1, column 1
Expected a key "orcid_id" or maybe "orcid-id" but it was not found. Maybe it was indented incorrectly? For reference, keys "orcid_id", "name", "private_key" and "public_key" are required here and "introduction_nanopub_uri" is optional, but only "name", "public_key" and "private_key" were given.
Your nanopub profile has not been set up yet, or is not set up correctly.

    Follow these instructions to correctly setup your nanopub profile:
    https://nanopublication.github.io/nanopub-py/getting-started/setup/#setup-your-profile


---
# 📋 JSON TEMPLATE

Create a JSON file with this structure:

```json
{
  "author": {
    "orcid": "0000-0000-0000-0000",
    "name": "Your Name"
  },
  "pico": {
    "title": "Your systematic review title",
    "population": "Who or what is being studied",
    "intervention": "What intervention or exposure",
    "comparison": "Comparison group (or 'Not applicable')",
    "outcome": "What outcomes are measured",
    "research_question": "Your full research question",
    "question_type": "effectiveness"
  },
  "output": {
    "filename": "my-pico"
  }
}
```

**Question types:** `causation`, `descriptive`, `effectiveness`, `experience`, `prediction`

**Note:** The `evaluation` type is NOT available in the Cochrane PICO template.

---